[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurergroup/ML_in_CP_UVie_SS26/blob/master/Module_II_Data_Featurization/WS2_Data_Representation/WS2_Data_Representation.ipynb)

# WS 2 — Data Representation and Featurisation

**Machine Learning in Computational Physics** · University of Vienna

---

<div class="alert alert-block alert-danger">

**Assignment submission**

When you submit this notebook for assignment, make sure that all tasks below (blue boxes) are completed (withc ode) and all questions are answered (with extra markdown responses). Make sure that the notebook runs correctly when all cells are executed in the right order from top to bottom. Before submission, do not clear outputs. Leave all the outputs from the last run included.
</div>

## 0. Prelude

In this workshop, we will be looking at how to represent atomistic structures with machine-learning-friendly representations, i.e. we will featurise molecules and materials.

We will use packages commonly used in computational materials modelling:
- [Atomic Simulation Environment (ASE)](https://wiki.fysik.dtu.dk/ase/): We will use this to store molecular structures and properties. ASE defines a database structure that is commonly used in machine learning workflows 
- [scikit-learn](https://scikit-learn.org/stable/): machine learning library
- [dscribe](https://singroup.github.io/dscribe/stable/): library to generate molecular representations (descriptors) with various descriptors
- [acepotentials.jl](https://github.com/ACEsuit/ACEpotentials.jl): package to create atomic cluster expansion representations and interatomic potentials, written in Julia, but we will use it with a python-wrap around function

As you go through the notebook and work through the tasks, look through the documentation pages of those packages if you get stuck.

We will be working on two datasets of molecules, one that explores composition space (dataset of  molecules with different elemental composition), and one that explores configurational space (data from molecular dynamics, i.e. same molecule but atoms in different positions):
1. Dataset of five short molecular dynamics trajectories of cyclohexane, Courtesy of "Quantum Chemistry in the Age of Machine Learning", edited by Pavlo Dral (2022)
2. The QM7 dataset of 7101 organic molecules with up to 7 non-hydrogen atoms, Courtesy of Rupp et al., Phys. Rev. Lett. 108, 058301 (2012) and qmlcode.org

Both datasets feature structures and energies.

In [ ]:
#basic stuff
import os
import sys
from functools import partial
import numpy as np
import random

import ase
from ase.io import read, write
from ase.visualize import view
from ase.build import molecule
from ase.db import connect

from matplotlib import pyplot as plt
import matplotlib as mpl
import pandas
import seaborn as sns
from weas_widget import WeasWidget

#ML stuff
import dscribe
import sklearn
from sklearn.metrics.pairwise import pairwise_kernels
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm # progress bars for loops

#acepotentials
import tempfile
from juliacall import Main as jl
jl.seval('import Pkg; Pkg.add("ACEpotentials")')
jl.seval('import Pkg; Pkg.add("ExtXYZ")')
%matplotlib inline

The following is a set of convenience functions, which we will use to build ACE descriptors.

## 1. Data Preparation and Analysis

Let's start with a dataset of five short MD simulations of different conformers of cyclohexane.

We start five independent molecular dynamics simulations
initialized within each of the known cyclohexane conformers shown in the below figure. The MD simulations were run for 10,000 time
steps each. The data contains the atom positions, velocities, energies and forces. These types of MD simulations will explore the energy landscape and settle within
local and/or global minima.

![cyclohexane conformers](../../graphics/conformers.png)

### Read and analyse Molecular Dynamics Data

The data can be found in the folder `/data/cyclohexane_data/md/` where each trajectory of structures sits in a file such as `chair.xyz`. 

<div class="alert alert-block alert-info">

**TASK**

1. Look at the files and study the XYZ file structure. In fact, this is an [extXYZ](https://www.ovito.org/docs/current/reference/file_formats/input/xyz.html#extended-xyz-format) file. What information is contained?
</div>

We will use the [ase.read](https://wiki.fysik.dtu.dk/ase/ase/io.html#ase.io.read) function to parse the data. It is very convenient for reading in xyz files (and many other common formats), and it can read in multiple frames at once. Here we read in all 10000 frames from each trajectory file at once, which is much faster than reading them in, one at a time.

Once the file is read, it creates a list of [ase.Atoms](https://ase-lib.org/ase/atoms.html) objects. This data structure is designed to hold many important definitions for materials (sum formula, positions, unit cell, etc.). The `Atoms` object is a central component of the `ase` package.

In [ ]:
# read in the frames from each MD simulation
traj = []
names = ['chair', 'twist-boat', 'boat', 'half-chair', 'planar']
rgb_colors = [(0.13333333333333333, 0.47058823529411764, 0.7098039215686275),
                (0.4588235294117647, 0.7568627450980392, 0.34901960784313724),
                (0.803921568627451, 0.6078431372549019, 0.16862745098039217),
                (0.803921568627451, 0.13725490196078433, 0.15294117647058825),
                (0.4392156862745098, 0.2784313725490196, 0.611764705882353),]

ranges = np.zeros((len(names), 2), dtype=int)
conf_idx = np.zeros(len(names), dtype=int)

for i, n in enumerate(names):
    # here comes the parsing of the files
    frames = read(f'./data/cyclohexane_data/MD/{n}.xyz', '::') #creates a list of Atoms objects.

    for frame in frames:
        # wrap each frame in its box, this means that we ensure that the coordinates of each atom are between 0 and the box size, which is important for the featurization step later on.
        frame.wrap(eps=1E-10)

        # mask each frame so that descriptors are only centered on carbon (#6) atoms
        mask = np.zeros(len(frame))
        mask[np.where(frame.numbers == 6)[0]] = 1 #the mask is list of 0s and 1s, where 1 indicates that the atom is a carbon atom and 0 indicates that it is not. This is used to tell the featurizer which atoms to center the descriptors on.
        frame.arrays['center_atoms_mask'] = mask

    #now we create three lists that contain the whole dataset. 
    
    # ranges is a list of tuples that indicate the start and end index of each trajectory in the traj list and traj is a list of all frames from all trajectories.
    ranges[i] = (len(traj), len(traj) + len(frames)) #list of data ranges

    conf_idx[i] = len(traj) # handy list to indicate the index of the first frame for each trajectory

    traj = [*traj, *frames] # full list of frames, concatenated to a list with 50000 entries

After this call, all the MD frames of the 5 different runs sit in the object `traj`. 
`traj` is a list of ASE `Atoms` objects, which contain the structure definition, positions, velocities, energies etc.

If you are unfamiliar with ASE, please explore the online documentation and play around a bit with it in the following cells.

You can see, we have 50,000 configurations, each labeled with an energy in electronvolt (eV).

In [ ]:
# play around here

print(traj[0]) #first frame
print(traj[0].get_positions())

**Task for you**
<div class="alert alert-block alert-info">

- Take some time to understand the data processing above.
- Visualize the trajectories of the 5 different molecules and study their different configurations and dynamics. Just change the name of the trajectory in the cell below
    
</div>

In [ ]:
#pick a molecule to visualize its trajectory
# ['chair', 'twist-boat', 'boat', 'half-chair', 'planar']
molecule = 'chair'
########
i = names.index(molecule)
r = ranges[i]
mol_traj = traj[r[0]:r[1]]

print("Visualizing '{0}' trajectory".format(molecule))
# visualize the trajectory of the molecule
viewer = WeasWidget()
viewer.from_ase(mol_traj)
viewer.avr.model_style = 1
viewer.avr.show_hydrogen_bonds = True
viewer


Next, we plot the energy as a function of time along the trajectory.

In [ ]:
# energies of the simulation frames
energy = np.array([a.info['energy_eV'] for a in traj])

# energies of the known conformers
c_energy = np.array([traj[c].info['energy_eV'] for c in conf_idx])

# extrema for the energies
max_e = max(energy)
min_e = min(energy)

print('energy range goes from {0:10.6f} to {1:10.6f} eV'.format(min_e, max_e))

In [ ]:
fig, ax = plt.subplots(1, figsize=(3*4.8528, 3*1.2219))

for n, c, r, rgb in zip(names, c_energy, ranges, rgb_colors):
    ax.plot(range(0, r[1] - r[0]),
            energy[r[0]:r[1]] - min_e,
            label=n,
            c=rgb,
            zorder=-1)
    
ax.legend()
ax.set_xlabel("Simulation Timestep")
ax.set_ylabel("Energy in eV")

ax.set_xlim([0, len(energy)//5])
ax.set_ylim([-0.1, 1.25 * (max_e - min_e)])
ax.set_yticklabels([])

plt.tight_layout()
plt.savefig('energy.png')
plt.show()

We can see that most trajectories appear relatively stable in their energy along the trajectory, but the 'planar' trajectory seems to change its energy (and configuration) relatively quickly to align with one of the others.


**Task for you**
<div class="alert alert-block alert-info">
    
- The five simulations converge into two different conformers, the chair conformer and the twist-boat conformer. Can you tell which simulation converges into which configuration? (This is doable the old-fashioned way by visually inspecting the MD simulations or by studying appropriate descriptors)

- Make a note of this as it will help you to rationalise some of the later findings.
</div>


In [ ]:
#your code here

### Read and analyse QM7 molecule dataset

The QM7 dataset contains 7101 molecules with up to 7 non-hydrogen atoms. It is one of the first datasets that was studied when the idea of "ab initio machine learning" came up, so machine learning regression of first principles properties based on feature representations that correlate well with molecular structure and properties.

For each of the 7101 molecules, we have 2 labels:
- PBE0/def2-TZVP atomization energy in eV (`energy_eV`)
- and the difference between PBE0 and DFTB3 atomization energy in eV (`delta_energy_eV`)

Reading the database can take a minute, so please be patient.

In [ ]:
def get_qm7_energies(filename, key="dft"):
    """ Returns a dictionary with heats of formation for each xyz-file.
    """

    f = open(filename, "r")
    lines = f.readlines()
    f.close()

    energies = []

    for line in lines:
        tokens = line.split()

        xyz_name = tokens[0]
        hof = float(tokens[1])
        dftb = float(tokens[2])

        if key=="delta":
            energies.append(hof - dftb)
        else:
            energies.append(hof)

    return energies


In [ ]:
# Import QM7, already parsed to QML
from ase.io import read

qm7_dft_energy = get_qm7_energies("./data/qm7/hof_qm7.txt", key="dft")
qm7_delta_energy = get_qm7_energies("./data/qm7/hof_qm7.txt", key = "delta")

qm7 = read(f'./data/qm7/qm7.xyz', '::') #creates a list of Atoms objects.


Let's visualize some of those more than 7000 molecules. You will quickly see that the database also contains some rather strange ones ...

In [ ]:
#just change the list index
mol = qm7[108]
print(mol)
viewer = WeasWidget()
viewer.from_ase(mol)
viewer.avr.model_style = 1
viewer.avr.show_hydrogen_bonds = True
viewer

In [ ]:
#lets attach the labels to the ASE objects as we did in the previous dataset
for i, mol in enumerate(qm7):
    mol.info["energy_eV"] = qm7_dft_energy[i]         # PBE0/def2-TZVP atomization energy in eV
    mol.info["energy_delta_eV"] = qm7_delta_energy[i] # difference between PBE0 and DFTB3 atomization energy in eV

random.seed(42)
random.shuffle(qm7)


Let's analyse this data. We first look at the molecular weight distribution and the elemental distribution

In [ ]:

mol_weights = np.array([a.get_masses().sum() for a in qm7], dtype=np.float64) # calculate molecular weight
number_of_atoms = np.array([len(a) for a in qm7], dtype=np.float64) # calculate number of atoms in each molecule



In [ ]:
import matplotlib.pyplot as plt
import scipy.stats as st

### plotting histograms

fig, axes = plt.subplots(1, 2, figsize=(10, 4))  # 1 row, 2 columns
data1, data2 = mol_weights, number_of_atoms

#Freedman-Diaconis rule to find optimal number of bins for histogram
q25, q75 = np.percentile(data1, [25, 75])
bin_width = 2 * (q75 - q25) * len(data1) ** (-1/3)
bins = round((data1.max() - data1.min()) / bin_width)
print("Freedman–Diaconis number of bins for molecular weight:", bins)

axes[0].hist(data1, bins=bins, density=True, alpha=0.7)#, histtype='step')
mn, mx = axes[0].get_xlim()
kde_xs = np.linspace(mn, mx, 300)
kde = st.gaussian_kde(data1)
axes[0].plot(kde_xs, kde.pdf(kde_xs), label="PDF")
axes[0].set_title('Molecular weight distribution')
axes[0].set_xlabel('Molecular weight')
axes[0].set_ylabel('Frequency')

q25, q75 = np.percentile(data2, [25, 75])
bin_width = 2 * (q75 - q25) * len(data2) ** (-1/3)
bins = round((data2.max() - data2.min()) / bin_width)
print("Freedman–Diaconis number of bins for molecule size:", bins)

# Plot the second histogram
axes[1].hist(data2, bins=bins, density=True, alpha=0.7)#, histtype='step')
mn, mx = axes[1].get_xlim()
kde_xs = np.linspace(mn, mx, 300)
kde = st.gaussian_kde(data2)
axes[1].plot(kde_xs, kde.pdf(kde_xs), label="PDF")
axes[1].set_title('Molecule size distribution')
axes[1].set_xlabel('Molecule size')
axes[1].set_ylabel('Frequency')

# Adjust layout and show the plot
plt.tight_layout()
plt.show()


**Task for you**
<div class="alert alert-block alert-info">

- Calculate and report the average molecular weight and number of atoms as well as the standard deviations of those two quantities    
- For the QM7 dataset, plot the PBE0 and PBE0-DFTB3 "delta" energy distributions as histograms. 
- Do molecular weight or molecule size correlate with the PBE0 atomization energy?

</div>


In [ ]:
##your code here

edata = np.array(qm7_dft_energy)
edata_delta = np.array(qm7_delta_energy)

### Data Prep Summary

Now we understand the basics of our datasets and have them set up. 

- The cyclohexane MD data arrays are
`names` (list of 5 conformation names), `ranges` (list of data ranges), `conf_idx` (handy list to indicate the index of the first frame for each trajectory), `traj` (full list of 50,000 structures)

- The QM7 data sits in `qm7`

## 2. Generate Descriptors




We will now play around with two different examples of descriptors to capture the key features of the dynamics. First, we will look at global descriptors and then at  atom-centered descriptors.

This will include
- Coulomb Matrix (global)
- MBTR (global)
- SOAP (can be used as atom-centered and global)
- Atomic Cluster Expansion (atom-centered and global)

### Coulomb Matrix


Coulomb Matrix (CM) is a simple global descriptor which mimics the
electrostatic interaction between nuclei.

Coulomb matrix is calculated with the equation below.

$$
    \begin{equation}
    M_{ij}^\mathrm{Coulomb}=\left\{
        \begin{matrix}
        0.5 Z_i^{2.4} & \text{for } i = j \\
            \frac{Z_i Z_j}{R_{ij}} & \text{for } i \neq j
        \end{matrix}
        \right.
    \end{equation}
$$

The diagonal elements can be seen as the interaction of an atom with itself. The strange exponent arises from an attempt to correlate the trend of the atomic energy w.r.t $Z_i$. If the diagonal elements can closely capture that trend, it goes along way in correctly capturing molecular stability. The off-diagonal elements represent the Coulomb repulsion between nuclei $i$ and $j$. Therefore, all elements of the representation are physically motivated.

In [ ]:
import collections.abc
collections.Iterable = collections.abc.Iterable # this is unfortunately necessary for compatbility of describe with Py>=3.10
from dscribe.descriptors import CoulombMatrix

atomic_numbers = [1, 6] # H and C atoms

# Setting up the CM descriptor
cm = CoulombMatrix(
    n_atoms_max=18, # maximum no. atoms in the studied molecules
    permutation = 'eigenspectrum', #takes the list of eigenvalues of the CM as the feature vector
    #permutation="sorted_l2"
)

In [ ]:
# Create CM output for the system
coulomb_matrices =cm.create(traj)
print("flattened", coulomb_matrices.shape)


**Task for you**
<div class="alert alert-block alert-info">
    
- Take a look at one of the CM descriptors for a given frame. Compare to the list of atoms. Does it make sense? 
    
</div>



In [ ]:
def matprint(mat, fmt="g", size=18):
    """
    Convenience function for pretty printing of matrices. Use it to print the Coulomb matrix.
    """
    if len(mat.shape)>1:
        #mat = mat.reshape([size,size])
        col_maxes = [max([len(("{:"+fmt+"}").format(x)) for x in col]) for col in mat.T]
        for x in mat:
            for i, y in enumerate(x):
                print(("{:"+str(col_maxes[i])+fmt+"}").format(y), end="  ")
            print("")
    else:
        words = ["{0:8.4f}".format(x) for x in mat]
        print(" ".join(words))




In [ ]:
# your code and answer here


# answers to questions



A good descriptor should be invariant with respect to atom permutations, translations, and rigid rotations. However, for the CM, it matters in which order the atoms appear. Changing of atom ordering leads to swapping of rows and columns in the matrix. This should raise alarm bells as it means that it isn't invariant to atom index permutations and therefore not suitable for general machine learning.

**Task for you**
<div class="alert alert-block alert-info">
    
- The CM construction above allows for multiple different options for `permutation`. Read the DScribe documentation and try different ones. Which one satisfies permutational invariance? This is the one we should be using.
</div>



In [ ]:
#Pick first molecule frame
first_frame = frames[0]
cm_original = cm.create(first_frame)
print('Original CM')
matprint(cm_original)

# Translation
first_frame.translate((5, 7, 9))
cm_translated = cm.create(first_frame)
print('Translated CM')
matprint(cm_translated)

# Rotation
first_frame.rotate(90, 'z', center=(0, 0, 0))
cm_rotated = cm.create(first_frame)
print("Rotated CM")
matprint(cm_rotated)

# Permutation
upside_down = first_frame[::-1]
cm_upside_down = cm.create(upside_down)
print("upside down CM")
matprint(cm_upside_down)

print('Differences due to translation')
trans_diff = (cm_translated-cm_original)
matprint(trans_diff)
print('Differences due to rotation')
rot_diff = (cm_rotated-cm_original)
matprint(rot_diff)
print('Differences due to permutation')
permut_diff = (cm_upside_down-cm_original)
matprint(permut_diff)

The benefit of a global descriptor is that it always has the same length, so we can directly compare between any frame. The downside is that it is not transferable across system sizes and it does not resolve individual atom/element configuration depdendencies.


In [ ]:
counts, bins = np.histogram(coulomb_matrices.flatten(),bins=200)
#plt.stairs(counts, bins)
plt.hist(bins[:-1], bins, weights=counts)
plt.xlabel('CM feature value')
plt.ylabel('log(count)')
plt.yscale('log')
plt.show()


**Task for you**
<div class="alert alert-block alert-info">
    
- What does the plot above tell us?    
- How does the CM construction work for the QM7 dataset where different molecules have different numbers of atoms? Generate CM features for the QM7 dataset and plot a few of them as a histogram

(Hint: You will need to figure out what elements are contained in the database and what the maximum number of atoms in the largest molecule is.)
</div>


In [ ]:
# space to play around

import collections.abc
collections.Iterable = collections.abc.Iterable # this is unfortunately necessary for compatbility of describe with Py>=3.10
from dscribe.descriptors import CoulombMatrix


max_atoms_qm7 = int(number_of_atoms.max())

atomic_numbers = []
for mol in qm7:
    atomic_numbers.extend(mol.get_atomic_numbers())

atomic_numbers = np.unique(np.array(atomic_numbers))
print(atomic_numbers)

species = []
for mol in qm7:
    species.extend(mol.get_chemical_symbols())

species = np.unique(np.array(species))
print(species)

# Setting up the CM descriptor
cm_qm7 = CoulombMatrix(
    n_atoms_max=max_atoms_qm7, # maximum no. atoms in the studied molecules
    #permutation = 'eigenspectrum',
    permutation="sorted_l2"
)

In [ ]:
# Create CM output for the system
coulomb_matrices_qm7 =cm_qm7.create(qm7)
print("flattened", coulomb_matrices_qm7.shape)


counts, bins = np.histogram(coulomb_matrices_qm7.flatten(),bins=200)
#plt.stairs(counts, bins)
plt.hist(bins[:-1], bins, weights=counts)
plt.xlabel('CM feature value')
plt.ylabel('log(count)')
plt.yscale('log')

plt.show()


### MBTR

The many-body tensor representation (MBTR) encodes a structure by using a distribution of different structural motifs. It can be used directly for both finite and periodic systems. MBTR is especially suitable for applications where interpretability of the input is important because the features can be easily visualized and they correspond to specific structural properties of the system (distances, angles, dihedrals).

The MBTR representation can be used as a global representation (one feature vector per system) or as an atom-centered or local one (one feature vector per atom). As a global descriptor, the individual contributions are discretised on a single spectrum for the molecule as shown in the below figure.

![bla](https://singroup.github.io/dscribe/latest/_images/mbtr.jpg)

In [ ]:
import collections.abc
collections.Iterable = collections.abc.Iterable # this is unfortunately necessary for compatbility of dscribe with Py>=3.10
from dscribe.descriptors import MBTR

min_dist = 0.8  #smallest distance in Angstrom
max_dist = 12.0  #largest distance in Angstrom
#we are constructing MBTR with inverse distances
grid_min = 1.0/max_dist # 1/Angstrom
grid_max = 1.0/min_dist # 1/Angstrom
grid_n = 100 # number of grid points
sigma = 0.05

#k=1, 1-body
#"atomic_number": The atomic number.
#k=2, 2-body
#"distance": Pairwise distance in angstroms.
#"inverse_distance": Pairwise inverse distance in 1/angstrom.
#k=3, 3-body
#"angle": Angle in degrees.
#"cosine": Cosine of the angle.

# MBTR setup for cyclohexane dataset
mbtr = MBTR(
    species=["H", "C"],
    geometry={"function": "distance"}, # only 2-body terms, no 3-body
    grid={"min": grid_min, "max": grid_max, "n": grid_n, "sigma": sigma},
    weighting={"function": "exp", "scale": 0.5, "threshold": 1e-3},
    periodic=False,
    normalization="l2",
    sparse=False
)

In [ ]:
#mbtrs = mbtr.create(traj)
#traj[0].momenta()

mbtrs = mbtr.create(traj[:10]) #calculate MBTR for first 10 frames
mbtrs = mbtr.create(traj) #calculate MBTR for all frames
print(mbtrs.shape)

In [ ]:
plt.plot(mbtrs[4]) #visualise 4th frame
plt.title('Full MBTR vector for one molecule')


# Create the mapping between an index in the output and the corresponding
# chemical symbol
n_elements = len(mbtr.species)
x = np.linspace(grid_min, grid_max, grid_n)

# Plot k=2
if mbtr.k==2:
    fig, ax = plt.subplots()
    for i in range(n_elements):
        for j in range(n_elements):
            if j >= i:
                i_species = mbtr.species[i]
                j_species = mbtr.species[j]
                loc = mbtr.get_location((i_species, j_species))
                plt.plot(x, mbtrs[0][loc], label="{}-{}".format(i_species, j_species))
    if mbtr.geometry['function'] == 'inverse_distance':
        ax.set_xlabel("Inverse distance (1/angstrom)")
    else:
        ax.set_xlabel("distance (angstrom)")
    plt.title('Individual components')

#plot k=3
if mbtr.k==3:
    fig, ax = plt.subplots()
    for i in range(n_elements):
        for j in range(n_elements):
            for k in range(n_elements):
                        i_species = mbtr.species[i]
                        j_species = mbtr.species[j]
                        k_species = mbtr.species[k]
                        loc = mbtr.get_location((i_species, j_species, k_species))
                        plt.plot(x, mbtrs[0][loc], label="{}-{}-{}".format(i_species, j_species, k_species))
    if mbtr.geometry['function'] == 'angle':
        ax.set_xlabel("angle in rad")
    else:
        ax.set_xlabel("cos(angle)")

    plt.title('Individual components')


ax.legend()
plt.show()

**Task for you**
<div class="alert alert-block alert-info">


- Can you explain why the MBTR descriptor has the length it does?   
- Is this the optimal distance range to be considered? What does the value of sigma do to the descriptor?
- What impact does it have to use 1-body, 2-body, 3-body components? How large is the descriptor in each case?
- Play around with MBTR parameters a bit to find the optimal settings, then revert to the original settings provided.

- Does MBTR satisfy all our symmetry requirements?

(Hint: pick a small set of geometries traj[0:10] to play around with settings to reduce computing time)

</div>

**Task for you**
<div class="alert alert-block alert-info">
    
- Set up an MBTR feature representation for the QM7 dataset and play around with the parameters available to you (see dscribe docs)
(Make sure to give the MBTR object a different name, e.g. mbtr_qm7)
</div>


In [ ]:
# space to play around

### SOAP

Smooth Overlap of Atomic Positions (SOAP) is a descriptor that encodes atomic environments by using a local expansion of a gaussian smeared atomic density with orthonormal functions based on spherical harmonics and radial basis functions.

$$
   p^{Z_1 Z_2}_{n n' l} = \pi \sqrt{\frac{8}{2l+1}}\sum_m {c^{Z_1}_{n l m}}^*c^{Z_2}_{n' l m}
$$

here $n$ and $n'$ are indices for the different radial basis
functions up to $n_\mathrm{max}$, $l$ is the angular degree of the
spherical harmonics up to $l_\mathrm{max}$ and $Z_1$ and $Z_2$
are atomic species.

The coefficients $c^Z_{nlm}$ are defined as the following inner
products:

$$
   c^Z_{nlm} =\iiint_{\mathcal{R}^3}\mathrm{d}V g_{n}(r)Y_{lm}(\theta, \phi)\rho^Z(\mathbf{r}).
$$

where $\rho^Z(\mathbf{r})$ is the gaussian smoothed atomic density for
atoms with atomic number $Z$ defined as

$$
   \rho^Z(\mathbf{r}) = \sum_i^{\lvert Z_i \rvert} e^{-1/2\sigma^2 \lvert \mathbf{r} - \mathbf{R}_i \rvert^2}
$$

$Y_{lm}(\theta, \phi)$ are the real spherical harmonics, and
$g_{n}(r)$ is the radial basis function.

For the radial degree of freedom, $g_{n}(r)$, multiple approaches may be used. By
default the DScribe implementation uses spherical gaussian type orbitals as
radial basis functions, as they allow  faster analytic computation.

The SOAP similarity kernel between two atomic environments can be retrieved
as a normalized polynomial kernel of the partial power spectrums:
$$
   K^\mathrm{SOAP}(\mathbf{p}, \mathbf{p'}) = \left( \frac{\mathbf{p} \cdot \mathbf{p'}}{\sqrt{\mathbf{p} \cdot \mathbf{p}~\mathbf{p'} \cdot \mathbf{p'}}}\right)^{\xi}
$$

Starting point for parameters:
- `r_cut=3.5`: we are considering 3.5A radius of sphere for describing each atom environments. For cyclohexane, this includes the whole ring, which is necessary to differentiate conformers. For QM7, this will likely be quite different.
- `n_max=4`: expand over 4 radial GTO bases
- `l_max=4`: expand over the first 4 spherical harmonics
- `sigma=0.3`: assume each atom has a gaussian of width sigma=0.3 imposed on its lattice site

SOAP is a descriptor centred around atoms. For the construction of machine learning interatomic potentials, this is beneficial. 

In [ ]:
import collections.abc
collections.Iterable = collections.abc.Iterable # this is unfortunately necessary for compatbility of describe with Py>=3.10
from dscribe.descriptors import SOAP

species = ["H", "C"]
r_cut = 3.5 #cutoff in Angstrom
n_max = 4 # 
sigma = 0.3 #stdev of gaussians
l_max = 4

# Setting up the SOAP descriptor
soap = SOAP(
    species=species,
    periodic=False,
    r_cut=r_cut,
    n_max=n_max,
    l_max=l_max,
    sigma=sigma,
)


In [ ]:
# Create SOAP output for all frames
soaps = soap.create(traj[:10]) #first 10 frames
#soaps = soap.create(traj)
print(soaps.shape)

We get a descriptor array with 3 dimensions: no. of frames, no. of atoms, and length of SOAP feature vector. Each atom has its own descriptor that represents the configuration of other atoms around this central atom.

In [ ]:
plt.plot(soaps[0,0,:]) #plot the soap vector for the first geometry, first atom
plt.title('Full SOAP feature vector for 1st geometry, 1st atom')

**Task for you**

<div class="alert alert-block alert-info">

- Visualise the SOAP feature vectors also for other atoms in the structurte (e.g. H atoms)  
- Play around with r_cut, n_max, l_max and study how the length of the descriptor changes. How do the input parameters change the length of the SOAP vectors?

- Show that the SOAP descriptors are also invariant w.r.t. to translations, permutations, and rotations
</div>

In [ ]:
# space to play around

Measuring the similarity of structures becomes easy when the feature vectors represent the whole structure, such as in the case of Coulomb matrix or MBTR. In these cases the similarity between two structures is directly comparable  with different kernels, e.g. the linear or Gaussian kernel.

Sometimes, we are focussed on comparative analysis between systems, then it is more useful to have a global descriptor. We can transform SOAP (and other atom-centered descriptors) into a global descriptor. We can achieve this by generating an average kernel. DScribe enables this with the option `average='inner'`.



In [ ]:
# Setting up the SOAP descriptor
soap = SOAP(
    species=species,
    periodic=False,
    r_cut=r_cut,
    n_max=n_max,
    l_max=l_max,
    sigma=sigma,
    average='inner', #this extra keyword makes it a global descriptor by averaging over all atoms in the molecule
)


In [ ]:
# Create SOAP output for all frames
soaps = soap.create(traj[:10])
#soaps = soap.create(traj)
print(soaps.shape)


Now we get only one SOAP descriptor per frame as we have averaged over the atoms

In [ ]:
plt.plot(soaps[0,:]) #plot the soap vector for the first geometry, first atom
plt.title('Full SOAP feature vector for 1st geometry, 1st atom')

In [ ]:
counts, bins = np.histogram(soaps.flatten(),bins=100)
#plt.stairs(counts, bins)
plt.hist(bins[:-1], bins, weights=counts)
plt.xlabel('SOAPS')
plt.ylabel('count')
plt.show()


**Tasks for you**
<div class="alert alert-block alert-info">
    
- When SOAP is an atom-centered descriptor, 3.5 Angstrom cutoff radius around each atom pretty much covers all relevant information for cyclohexane. When we average and create a global descriptor, is this still the case? 
- Try to generate SOAP descriptors for the QM7 dataset. We will need this later.

</div>

In [ ]:
#space to play around

We will now generate several different SOAP descriptors for the cyclohexane dataset

In [ ]:
#Let's generate two different soap descriptors with different settings so we hvae them handy
import collections.abc
collections.Iterable = collections.abc.Iterable # this is unfortunately necessary for compatbility of describe with Py>=3.10
from dscribe.descriptors import SOAP

####DEFAULT SOAP

species = ["H", "C"]
r_cut = 3.5 #cutoff in Angstrom
n_max = 4 # 
sigma = 0.3 #stdev of gaussians
l_max = 4

# Setting up the SOAP descriptor
soap = SOAP(
    species=species,
    periodic=False,
    r_cut=r_cut,
    n_max=n_max,
    l_max=l_max,
    sigma=sigma,
    average='inner',
)

soaps_default = soap.create(traj)

####SOAP with high n and l

species = ["H", "C"]
r_cut = 3.5 #cutoff in Angstrom
n_max = 6 # 
sigma = 0.3 #stdev of gaussians
l_max = 6

# Setting up the SOAP descriptor
soap = SOAP(
    species=species,
    periodic=False,
    r_cut=r_cut,
    n_max=n_max,
    l_max=l_max,
    sigma=sigma,
    average='inner',
)

soaps_high = soap.create(traj)

#### SOAP with even higher n and l and larger cutoff

#soap_big = SOAP(
#    species=species,
#    periodic=False,
#    r_cut=4.0,
#    n_max=8,
#    l_max=6,
#    sigma=0.3,
#    average='inner',
#)

#soaps_big=soap.create(traj)#, n_jobs=6) # we can parallelize over cores with n_jobs to speed things up

**Tasks for you**
<div class="alert alert-block alert-info">
    
- Which of the  descriptors with default parameters is likely to be the best? Rank them by their computational expense for the ML tasks that follow below?
- Before moving on, please go back to CM and MBTR and make sure you reset all values to default and generate descriptors for the full cyclohexane dataset (for all 50000 frames) and the QM7 dataset. 

</div>

## 3. Similarity Analysis

We can use our new descriptors to find the configurational change of the planar configuration without knowing anything about the structure. In the following box, we use a pairwise kernel from scikit-learn to compare structures for their similarity. For each trajectory, we compare the last frame with all other frames. We have the CM, the MBTR (inverse distance), and the three SOAP vectors to compare initially. Of course, you can play around with settings of all descriptors to find even better representations.

The ability of a descriptor to measure "distances" in the space of the dataset is a crucial prerequisite for being able to build reliable machine learning models.

In [ ]:
#descriptors = soaps_default
#descriptors = soaps_high
#descriptors = coulomb_matrices
descriptors = mbtrs

#descriptors = soaps_big

similarities = np.zeros((len(traj), 5))
#uses a kernel similary measure

kernel = partial(pairwise_kernels, metric='rbf', gamma=None) #allows us to call pairwise_kernel like a function
#kernel = partial(pairwise_kernels, metric='rbf', gamma=2.0) #allows us to call pairwise_kernel like a function
#You can also manually set the gamma of the kernel to control the sensitivity of the similarity measure. A smaller gamma will make the kernel more sensitive to differences in the descriptors, while a larger gamma will make it less sensitive. You can experiment with different gamma values to see how it affects the similarities. If you set "gamma=None", the gamma will be set to 1/n_features, which is a common default value.

ranges2 = ranges.copy()
ranges2[4,1] = -1
for i, ci in enumerate(tqdm(conf_idx)):
    ki = kernel(descriptors[ranges2[i,0]].reshape(1,-1), descriptors)
    similarities[:, i] = ki



In [ ]:
fig, ax = plt.subplots(1, figsize=(3*4.8528, 3*1.2219))

for i, (n, c, r, rgb) in enumerate(zip(names, c_energy, ranges, rgb_colors)):
    ax.plot(range(0, r[1] - r[0]),
            similarities[r[0]:r[1],i],
            label=n,
            c=rgb,
            zorder=-1)
    
ax.legend()
ax.set_xlabel("Simulation Timestep")
ax.set_ylabel("Similarity")

ax.set_xlim([0, len(energy)//5])
#ax.set_ylim([-0.1, 1.25 * (max_e - min_e)])
#ax.set_yticklabels([])

plt.tight_layout()
plt.savefig('similarity_soap_big.png')
plt.show()

A low similarity means that the descriptor considers the structures to be maximally unsimilar. 

- As we can see, the coulomb matrix is not good at identifying similarity. (See what happens with 'sorted_l2' and 'eigenspectrum'. What is the origin of the difference?)
- MBTR and SOAP do a lot better. 
- Using the SOAP descriptors, we see that all configurations retain similarity with the initial structure during the dynamics except for the planar structure, which starts in a structure that is disimilar to the final frame.

**Task for you**
<div class="alert alert-block alert-info">
    
- Go back up and retry this with the CM and MBTR descriptors with different settings. Then try tuning the SOAP with higher n_max and l_max. Can you improve the similarity description with MBTR and SOAP? Which descriptor performs best? The similarity difference between the initial and final structure of the planar geometry is a good indicator of the "resolving power" of the descriptor. Before we do any machine learning, we should make sure that the resolving power and the structural covariance that our descriptor captures is as large as possible. The descriptor that achieves the biggest disimilarity between different structures should be our descriptor of choice.
- do the similarity comparison based on the final structure in the MD (rather than the first). Can you identify which final geometries the different MD trajectories reach?
</div>

In [ ]:
#space to play

## 4. Homework Assignment

**Task for you**
<div class="alert alert-block alert-info">

1. Use the Atomic Cluster Expansion descriptor covered in the lecture (see code below)  
2. Compare SOAP and ACE for the cyclohexane trajectory data. Which of the two descriptors with default settings and the same cutoff better resolve similarity/dissimilarity of the structures visited during the dynamics?
3. Optimise the hyperparameters for SOAP and ACE (increase r_cut etc.) to achieve optimal dissimiliarity for both.
4. Use the optimised settings for SOAP and ACE to analyse the QM7 dataset. Identify the two molecules that are most disimilar and the two molecules that are most similar. Visualise the identified molecuels. Do the two descriptors give different answers?
5. Note that different descriptors may require different gamma values in the Gaussian similarity kernel for optimal performance
</div>

In [ ]:
#space to play

In [ ]:
def init_ace():
    """Load ACEpotentials in Julia (slow on first call, ~30s)."""
    jl.seval("using ACEpotentials")
    jl.seval("using ExtXYZ")

def compute_ace_descriptors(atoms_list, elements, rcut=5.5, order=3, totaldegree=8):
    """
    Compute ACE structural descriptors for a list of ASE Atoms. Creates a list of descriptor arrays, one per structure, where each array has shape (n_atoms, n_features).

    Parameters
    ----------
    atoms_list  : list of ase.Atoms
    elements    : list of str, e.g. ['C'] or ['Si', 'O']
    rcut        : float, cutoff radius in Å
    order       : int, body-order minus 1
    totaldegree : int, total polynomial degree

    Returns
    -------
    list of len(n_structures) of np.ndarray of shape (n_atoms, n_features)
    """
    import os
    from pathlib import Path

    # mkstemp returns a closed fd — avoids any handle held open by the juliacall Julia process
    fd, tmpfile = tempfile.mkstemp(suffix=".extxyz")
    os.close(fd)

    write(tmpfile, atoms_list, format="extxyz")

    elems = "[" + ", ".join(f":{e}" for e in elements) + "]"

    result = jl.seval(f"""
        using ACEpotentials
        using ExtXYZ                  
        dataset = ExtXYZ.load(\"{tmpfile}\")
        model = ace1_model(
            elements = {elems},
            rcut     = {rcut},
            order    = {order},
            totaldegree = {totaldegree},
        )
    #descriptors = [struct_descriptor = sum(site_descriptors(system, model)) / length(system) for system in #dataset]
    #permutedims(reduce(hcat, descriptors))   # → Matrix, rows = structures

    descriptors = [struct_descriptor = site_descriptors(system, model) for system in dataset]
    #permutedims(reduce(hcat, descriptors))   # → Matrix, rows = structures
    descriptors = [permutedims(reduce(hcat, descriptor)) for descriptor in descriptors]
    
    """)
    return list(result)

def compute_global_ace_descriptors(atoms_list, elements, rcut=5.5, order=3, totaldegree=8):
    """
    Compute global (per-structure) ACE descriptors for a list of ASE Atoms. Here, the descriptors of each atom are averaged to yield a single descriptor vector per structure.

    Parameters
    ----------
    atoms_list  : list of ase.Atoms
    elements    : list of str, e.g. ['C'] or ['Si', 'O']
    rcut        : float, cutoff radius in Å
    order       : int, body-order minus 1
    totaldegree : int, total polynomial degree

    Returns
    -------
    np.ndarray of shape (n_structures, n_features)
    """
    import os
    from pathlib import Path

    # mkstemp returns a closed fd — avoids any handle held open by the juliacall Julia process
    fd, tmpfile = tempfile.mkstemp(suffix=".extxyz")
    os.close(fd)

    write(tmpfile, atoms_list, format="extxyz")

    elems = "[" + ", ".join(f":{e}" for e in elements) + "]"

    result = jl.seval(f"""
        using ACEpotentials
        using ExtXYZ                  
        dataset = ExtXYZ.load(\"{tmpfile}\")
        model = ace1_model(
            elements = {elems},
            rcut     = {rcut},
            order    = {order},
            totaldegree = {totaldegree},
        )
    descriptors = [struct_descriptor = sum(site_descriptors(system, model)) / length(system) for system in dataset]
    permutedims(reduce(hcat, descriptors))   # → Matrix, rows = structures
    
    """)
    return np.array(result, dtype=np.float32)

Here are  examples how to generate local and global ACE descriptors:

In [ ]:
init_ace()   # once per kernel session

from ase.build import molecule
my_ase_dataset = [molecule("CH4"), molecule("C6H6"), molecule("C2H4")]

X = compute_ace_descriptors(
    atoms_list  = my_ase_dataset,
    elements    = ["C", "H"],
    rcut        = 3.5,
    order       = 3,
    totaldegree = 8,
)
print(X[1].shape)  # (n_structures, n_ace_features)


X = compute_global_ace_descriptors(
    atoms_list  = my_ase_dataset,
    elements    = ["C", "H"],
    rcut        = 3.5,
    order       = 3,
    totaldegree = 8,
)
print(X.shape)  # (n_structures, n_ace_features)

In [ ]:
#set up similarity for ACE
descriptors = aces

similarities = np.zeros((len(traj), 5))
#uses a kernel similary measure
kernel = partial(pairwise_kernels, metric='rbf', gamma=None) #allows us to call pairwise_kernel like a function

ranges2 = ranges.copy()
ranges2[4,1] = -1
for i, ci in enumerate(tqdm(conf_idx)):
    ki = kernel(descriptors[ranges2[i,0]].reshape(1,-1), descriptors)
    similarities[:, i] = ki